[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quantum/two_spins/two_spins.ipynb)

# Two Spins

Two spins, one up and one down, interact by exchange: over time they swap, and halfway they are entangled. Then neither spin on its own points anywhere, yet measured along the same direction the two always disagree. This notebook holds both spins in one geometric algebra, two copies of space. Two multivectors split every state of the pair into its singlet and triplet parts. What one spin shows on its own is the part of the pair's spin in its own planes. What the two show together is a map between their spaces, whose singular values decide how far the pair can break Bell's inequality, and one multivector identity decides how far any pair can.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np
from IPython.display import Image, display

from numga import Algebra, NumpyContext
from examples.animation import save_animation
from examples.quantum.two_spins import render

np.set_printoptions(precision=4, suppress=True)

# One spin in x y z, the other in X Y Z.
ga = Algebra("x+y+z+X+Y+Z+")
context = NumpyContext(ga)
mv = context.multivector
one = mv.scalar([1.0])                                                          # [] Scalar
First = ga.gatype(ga.subspace("x y z"))
Second = ga.gatype(ga.subspace("X Y Z"))
FirstPlanes = ga.gatype(ga.subspace("yz zx xy"))
SecondPlanes = ga.gatype(ga.subspace("YZ ZX XY"))
# The state of one spin, and of the pair: a product of the two spins' states.
FirstSpinor = ga.gatype(ga.subspace("1 yz zx xy"))
SecondSpinor = ga.gatype(ga.subspace("1 YZ ZX XY"))
Spinor = ga.gatype((FirstSpinor * SecondSpinor).output_subspace)
Correlation = ga.gatype((First, Second))

## 1. Two spins in one algebra

The state of one spin is an even multivector of its own space, as in the Pauli algebra: spin up along z is one, and spin down is the half turn that takes z to -z, `-zx`. The state of the pair is a product of the two, taken in the ideal of the correlator `correlator = 0.5 * (1 - xy * XY)`. In that ideal, multiplying on the right by either spin's xy plane is the same: it is `imaginary = correlator * xy`, which squares to `-correlator` and is the pair's imaginary unit. Planes of different spins commute, so each spin's planes act on the pair without disturbing the other's. A state is normalized to one in the scalar part of its density, `2 * (state >> one)`.

In bra-ket notation the pair's state reads as a ket $|\psi\rangle \in \mathbb{C}^2 \otimes \mathbb{C}^2$, spin up and spin down as $|0\rangle$ and $|1\rangle$, and right multiplication by `imaginary` as multiplication by $i$.

In [ ]:
correlator = 0.5 * (one - mv.xy * mv.XY)                                        # [] Spinor
imaginary = correlator * mv.xy                                                  # [] Spinor
# The first spin up, the second down.
up_down = -mv.ZX * correlator                                                   # [] Spinor
norm = 2 * (up_down >> one).select[0]                                           # [] Scalar

In [ ]:
print("correlator * correlator - correlator:", np.abs((correlator * correlator - correlator).kernel).max())
print("correlator * xy - correlator * XY:", np.abs((correlator * mv.xy - correlator * mv.XY).kernel).max())
print("imaginary * imaginary + correlator:", np.abs((imaginary * imaginary + correlator).kernel).max())
print("xy * XY - XY * xy:", np.abs((mv.xy * mv.XY - mv.XY * mv.xy).kernel).max(), "  norm:", norm.to_array())

## 2. What each spin shows, and what the two show together

The spin of the pair is `2 * (state >> imaginary)`, a bivector. What one spin shows by itself is the part of it in that spin's own planes, read as a vector of its space through its pseudoscalar: the spin's Bloch vector, of length one when the spin is in a state of its own, and shorter when the pair is entangled. What the two spins show together are their correlations along a direction of each: a map from the second spin's directions to the first's, read off the part of the density `2 * (state >> one)` that spans a plane of each spin, with `a | correlation(b)` the expected product of the two spins' values along a and b. The picture shows each Bloch vector in its ball, and between them the image of the second spin's unit directions under the correlation map.

The Bell combination of four correlations, along two directions of each spin, stays within 2 in any account in which each spin carries its own answers. For a given state, its largest value over all directions is `2 * sqrt(s1**2 + s2**2)`, from the two largest singular values of the correlation map. Spin up and spin down reach 2 exactly.

In bra-ket notation, with the Pauli matrices $\boldsymbol\sigma$, the Bloch vector of one spin reads as $\operatorname{tr}(\rho_1 \boldsymbol\sigma)$ of its reduced density matrix $\rho_1 = \operatorname{tr}_2 |\psi\rangle\langle\psi|$, the correlation map as the correlation tensor $T_{ij} = \langle\psi| \sigma_i \otimes \sigma_j |\psi\rangle$, and the largest Bell combination as $2\sqrt{t_1^2 + t_2^2}$ over the singular values $t_i$ of $T$.

In [ ]:
def bloch(state: Spinor):
    """Each spin's Bloch vector: the part of the spin in its own planes, read as a vector of its space."""
    spin = 2 * (state >> imaginary)                                             # [...] Bivector
    first = mv.xyz.inverse() * spin.cast(FirstPlanes)                           # [...] First
    second = mv.XYZ.inverse() * spin.cast(SecondPlanes)                         # [...] Second
    return first.cast(First), second.cast(Second)


def correlation(state: Spinor) -> Correlation:
    """The correlations of the two spins, as a map from the second spin's directions to the first's."""
    density = 2 * (state >> one)                                                # [...] Spinor
    return (mv.xyz.inverse() * (density * (mv.XYZ * Second)).cast(FirstPlanes)).cast(Correlation)


def bell(correlations: Correlation):
    """The largest Bell combination over all directions, from the two largest singular values."""
    values = correlations.svdvals()                                             # [..., 3] Scalar
    return 2 * (values[..., 0] ** 2 + values[..., 1] ** 2).square_root()       # [...] Scalar


start_first, start_second = bloch(up_down)                                      # [] First, Second
start_correlations = correlation(up_down)                                       # [] Correlation
start_bell = bell(start_correlations)                                           # [] Scalar
render.draw_state(start_first, start_second, start_correlations);

In [ ]:
print("largest Bell combination of spin up and spin down:", start_bell.to_array())

## 3. The exchange

The exchange interaction couples each plane of one spin to the same plane of the other: `coupling = yz * YZ + zx * ZX + xy * XY`. Squared, `coupling * coupling == 3 + 2 * coupling`: the coupling satisfies a quadratic with roots 3 and -1. So `(1 + coupling) / 4` and `(3 - coupling) / 4` are idempotent and add up to one. They split every state into its singlet part, on which the coupling is 3, and its triplet part, on which it is -1. Over time each part turns by its own rotor in the xy plane, multiplied on the right: the singlet part by three times the angle, the triplet part back by the angle. Spin up and spin down are half singlet and half triplet. As the two parts turn apart, the spins swap, and halfway they are fully entangled: neither shows a Bloch vector, the correlation map turns every unit direction into its opposite, and the largest Bell combination reaches `2 * sqrt(2)`.

In bra-ket notation the exchange reads as $e^{-i\theta\,\boldsymbol\sigma_1\cdot\boldsymbol\sigma_2}$, and the two idempotents as the projectors $\tfrac14\left(1 - \boldsymbol\sigma_1\cdot\boldsymbol\sigma_2\right)$ onto the singlet and $\tfrac14\left(3 + \boldsymbol\sigma_1\cdot\boldsymbol\sigma_2\right)$ onto the triplet.

In [ ]:
coupling = mv.yz * mv.YZ + mv.zx * mv.ZX + mv.xy * mv.XY                        # [] Spinor
singlet_part = 0.25 * (one + coupling)                                          # [] Spinor
triplet_part = 0.25 * (3 - coupling)                                            # [] Spinor


def exchange(state: Spinor, angle: np.ndarray) -> Spinor:
    """The state after the exchange has acted for the given angle, the coupling strength times time."""
    return singlet_part * state * (mv.xy * (3 * angle)).exp() + triplet_part * state * (mv.xy * -angle).exp()   # [...] Spinor


angles = np.linspace(0.0, np.pi / 4, 121)
states = exchange(up_down, angles)                                              # [angles] Spinor
first, second = bloch(states)                                                   # [angles] First, Second
correlations = correlation(states)                                              # [angles] Correlation
bells = bell(correlations)                                                      # [angles] Scalar
display(Image(filename=save_animation(render.animate_pair(angles, first, second, correlations, bells), "two_spins", 60)))

In [ ]:
print("coupling * coupling - 3 - 2 * coupling:", np.abs((coupling * coupling - 3 - 2 * coupling).kernel).max())
print("singlet part idempotent:", np.abs((singlet_part * singlet_part - singlet_part).kernel).max(), "  triplet part idempotent:", np.abs((triplet_part * triplet_part - triplet_part).kernel).max())
print("largest Bell combination at the start, halfway and the end:", bells.to_array()[[0, 60, -1]])

## 4. The most any pair can do

The singlet part of spin up and spin down, normalized, is the singlet: the one state on which the coupling is 3, which the exchange only turns by a phase. Its correlation map turns every direction of the second spin into the opposite direction of the first. How far can any state push the Bell combination? For given directions the combination is the expectation of a multivector acting on states from the left, the Bell element: each product of a plane of one spin with a plane of the other, `(I * a) * (I * b)`, is minus the product of the two spins' values along a and b. Squared, the Bell element is `4 - 4 * (a ^ a') * (b ^ b')`: four, less four times the product of the plane spanned by the first spin's two directions and the plane spanned by the second's. Neither plane is larger than one, so the square is at most eight, and no state takes the combination past `2 * sqrt(2)`. The singlet reaches it, with the first spin's directions a quarter turn apart and the second spin's halfway between them, reversed.

In bra-ket notation the Bell element reads as the CHSH operator $\mathcal{B} = A \otimes (B + B') + A' \otimes (B - B')$, its square as $\mathcal{B}^2 = 4 - [A, A'] \otimes [B, B']$, and the bound as Tsirelson's.

In [ ]:
def bell_element(first, first_other, second, second_other) -> Spinor:
    """The Bell combination along two directions of each spin, as a multivector acting on states from
    the left."""
    return -((mv.xyz * first) * (mv.XYZ * (second + second_other))
             + (mv.xyz * first_other) * (mv.XYZ * (second - second_other)))    # [...] Spinor


def expectation(element: Spinor, state: Spinor):
    """The expectation of a multivector acting on states from the left: the sandwich from the other
    side, `state.reverse() * element * state`."""
    return 2 * (state << element).select[0]                                     # [...] Scalar


singlet = np.sqrt(2) * singlet_part * up_down                                   # [] Spinor
singlet_correlations = correlation(singlet)                                     # [] Correlation
# Any four unit directions, and the Bell element's square against the product of the two planes.
rng = np.random.default_rng(0)
a, a_other = mv(First, rng.normal(size=(2, 3))).normalized()                    # [] First each
b, b_other = mv(Second, rng.normal(size=(2, 3))).normalized()                   # [] Second each
square = bell_element(a, a_other, b, b_other) * bell_element(a, a_other, b, b_other)   # [] Spinor
bound = 4 - 4 * (a ^ a_other) * (b ^ b_other)                                   # [] Spinor
# The directions at which the singlet reaches the bound.
half = 1 / np.sqrt(2)
reached = expectation(bell_element(mv.x, mv.y, -half * (mv.X + mv.Y), -half * (mv.X - mv.Y)), singlet)   # [] Scalar

In [ ]:
print("coupling * singlet - 3 * singlet:", np.abs((coupling * singlet - 3 * singlet).kernel).max())
print("correlation of X, Y, Z plus x, y, z:", [np.abs((singlet_correlations(across) + along).kernel).max() for across, along in ((mv.X, mv.x), (mv.Y, mv.y), (mv.Z, mv.z))])
print("square of the Bell element less its bound:", np.abs((square - bound).kernel).max())
print("Bell combination the singlet reaches:", reached.to_array(), "  2 * sqrt(2):", 2 * np.sqrt(2))

In [ ]:
# checks
# The correlator is idempotent and makes the two xy planes the same unit on the right; the singlet and
# triplet parts are idempotent and add up to one; the exchange keeps the state normalized and swaps the
# spins through a fully entangled state; the Bell element squares to four less four times the product
# of the two planes, and the singlet reaches the bound this sets.
np.testing.assert_allclose((correlator * correlator - correlator).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((correlator * mv.xy - correlator * mv.XY).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((coupling * coupling - 3 - 2 * coupling).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((singlet_part + triplet_part - one).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose(expectation(one, states).to_array(), 1.0, atol=1e-7)
np.testing.assert_allclose(first[60].kernel, 0.0, atol=1e-7)
np.testing.assert_allclose(bells.to_array()[[0, 60, -1]], [2.0, 2 * np.sqrt(2), 2.0], atol=1e-7)
np.testing.assert_allclose((first[-1] + mv.z).kernel, 0.0, atol=1e-7)
np.testing.assert_allclose((square - bound).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose(reached.to_array(), 2 * np.sqrt(2), atol=1e-12)